In [4]:
# Import the libraries needed for data preparation
import pandas as pd
import ast
import numpy as np
import re

In [5]:
# Load the Food.com recipe dataset
df = pd.read_csv("../recipe_10000.csv")

In [6]:
# Keep only the fields needed for generation and check for missing values
df = df[["RecipeIngredientParts", "RecipeInstructions"]]
print(df.shape)
print(df.isna().sum())
df.head(2)

(10000, 2)
RecipeIngredientParts    0
RecipeInstructions       0
dtype: int64


,RecipeIngredientParts,RecipeInstructions
0,"['blueberries', 'granulated sugar', 'vanilla y...","['Toss 2 cups berries with sugar.', 'Let stand..."
1,"['saffron', 'milk', 'hot green chili peppers',...",['Soak saffron in warm milk for 5 minutes and ...


In [7]:
# Define helper functions to parse list-like text and clean whitespace and HTML
def parse_list(value):
    if not isinstance(value, str) or not value.strip():
        return []
    value = value.strip()
    try:
        parsed = ast.literal_eval(value)
        if isinstance(parsed, (list, tuple)):
            return [str(x) for x in parsed]
        return [str(parsed)]
    except (ValueError, SyntaxError):
        pieces = re.findall(r'"([^"]*)"|\'([^\']*)\'', value)
        return [a or b for a, b in pieces]


def clean_text(s):
    s = str(s)
    s = re.sub(r"<[^>]+>", " ", s)
    s = s.replace("\u00a0", " ")
    s = re.sub(r"\s+", " ", s)
    return s.strip()


# Quick test on the first recipe
print(parse_list(df["RecipeIngredientParts"].iloc[0]))
print(clean_text("  Hello   <b>world</b>\n  "))

['blueberries', 'granulated sugar', 'vanilla yogurt', 'lemon juice']
Hello world


In [8]:
# Reload the data, clean every recipe, and drop the raw Name column
df = pd.read_csv("../recipe_10000.csv", usecols=["Name", "RecipeIngredientParts", "RecipeInstructions"])
df = df.dropna().reset_index(drop=True)

df["title"] = df["Name"].apply(clean_text)
df["ingredients"] = df["RecipeIngredientParts"].apply(
    lambda v: [clean_text(x) for x in parse_list(v) if clean_text(x)]
)
df["steps"] = df["RecipeInstructions"].apply(
    lambda v: [clean_text(x) for x in parse_list(v) if clean_text(x)]
)

# Keep only rows that have a title, at least one ingredient and at least one step
df = df[(df["title"] != "") & (df["ingredients"].str.len() > 0) & (df["steps"].str.len() > 0)]
df = df.reset_index(drop=True)

# Remove the raw Name column because the cleaned title column replaces it
df = df.drop(columns=["Name"])

print("Recipes after cleaning:", len(df))
print(df.columns.tolist())
print(df["title"].iloc[0])
print(df["ingredients"].iloc[0])
print(df["steps"].iloc[0][:2])

Recipes after cleaning: 9980
['RecipeIngredientParts', 'RecipeInstructions', 'title', 'ingredients', 'steps']
Low-Fat Berry Blue Frozen Dessert
['blueberries', 'granulated sugar', 'vanilla yogurt', 'lemon juice']
['Toss 2 cups berries with sugar.', 'Let stand for 45 minutes, stirring occasionally.']


In [9]:
df = df.drop(columns=["RecipeIngredientParts", "RecipeInstructions"])

In [10]:
# Remove duplicates, filter by instruction length, and pick a manageable training subset
SUBSET_SIZE = 8000
MIN_INSTR_CHARS = 80
MAX_INSTR_CHARS = 1500
SEED = 42

print("Start:", len(df))

# Remove recipes with the same title, then recipes with identical ingredients and steps
df = df.drop_duplicates(subset="title")
key = df["ingredients"].apply(lambda x: "|".join(x)) + "##" + df["steps"].apply(lambda x: "|".join(x))
df = df[~key.duplicated()]
print("After removing duplicates:", len(df))

# Keep recipes whose total instruction length is not too short or too long
instr_len = df["steps"].apply(lambda s: len(" ".join(s)))
df = df[(instr_len >= MIN_INSTR_CHARS) & (instr_len <= MAX_INSTR_CHARS)]
print("After length filter:", len(df))

# Take a random subset so training stays manageable
df = df.sample(n=min(SUBSET_SIZE, len(df)), random_state=SEED).reset_index(drop=True)
print("Final subset:", len(df))
df.head(3)

Start: 9980
After removing duplicates: 9632
After length filter: 9334
Final subset: 8000


,title,ingredients,steps
0,Delicious Baked Beans,"[bacon, onions, garlic, ginger, great northern...","[Preheat oven to 350 degrees., Cook bacon in l..."
1,Asparagus &amp; Tomato Salad,"[asparagus, tomatoes, fresh basil, sea salt, f...",[Blanch asparagus briefly in boiling water on ...
2,Classic Cabbage Rolls,"[cabbage, onion, butter, Italian stewed tomato...","[In a Dutch oven, boil cabbage for 10 minutes ..."


In [11]:
# Decode HTML entities such as &amp; and &#39; in titles, ingredients and steps
import html

entity_pattern = r"&#?\w+;"

# Count how many titles contain entities before the fix
print("Titles with entities before:", df["title"].str.contains(entity_pattern).sum())

df["title"] = df["title"].apply(html.unescape)
df["ingredients"] = df["ingredients"].apply(lambda items: [html.unescape(i) for i in items])
df["steps"] = df["steps"].apply(lambda items: [html.unescape(s) for s in items])

# Check that no entities remain anywhere
leftover_steps = df["steps"].apply(lambda s: bool(re.search(entity_pattern, " ".join(s)))).sum()
leftover_ing = df["ingredients"].apply(lambda s: bool(re.search(entity_pattern, " ".join(s)))).sum()
print("Titles with entities after:", df["title"].str.contains(entity_pattern).sum())
print("Recipes with entities left in ingredients:", leftover_ing, "| in steps:", leftover_steps)
print(df["title"].iloc[1])

Titles with entities before: 154
Titles with entities after: 0
Recipes with entities left in ingredients: 0 | in steps: 0
Asparagus & Tomato Salad


In [13]:
# Format each recipe into one consistent text block: Title, Ingredients, Instructions
def format_recipe(title, ingredients, steps):
    ing = "\n".join(f"- {i}" for i in ingredients)
    ins = "\n".join(f"{n}. {s}" for n, s in enumerate(steps, 1))
    return f"Title: {title}\nIngredients:\n{ing}\nInstructions:\n{ins}"

df["text"] = [format_recipe(t, i, s) for t, i, s in zip(df["title"], df["ingredients"], df["steps"])]

# Look at one formatted recipe and check the text lengths
print(df["text"].iloc[0])
print("\nRecipes:", len(df))
print(df["text"].str.len().describe())

Title: Delicious Baked Beans
Ingredients:
- bacon
- onions
- garlic
- ginger
- great northern beans
- brown sugar
- molasses
- barbecue sauce
- ketchup
- Worcestershire sauce
- prepared mustard
- cider vinegar
Instructions:
1. Preheat oven to 350 degrees.
2. Cook bacon in large skillet until fat is rendered and lightly brown, about 5 minutes.
3. Drain fat; save 2 tbsp.
4. fat and cook onion, garlic, and ginger until onion is tender, about 5 minutes.
5. Remove from heat.
6. Stir in beans, sugar, molasses, bbq sauce, ketchup, Worcestershire sauce, mustard, and vinegar.
7. Season to taste with salt and pepper.
8. Place bean mixture in an oven-proof baking dish or casserole; bake at 350 degrees for 30 minutes or until thickened.
9. Alternatively, they can be cooked on a bbq grill, away from the flames or coals, approximately 30 minutes, or until thick.

Recipes: 8000
count    8000.000000
mean      659.890500
std       291.275842
min       169.000000
25%       446.000000
50%       601.00000

In [14]:
# Save the final processed dataset to the shared data folder
from pathlib import Path

# Decoding entities can leave non-breaking spaces behind, so replace them with normal spaces
df["text"] = df["text"].str.replace("\u00a0", " ", regex=False)

DATA_DIR = Path("../data")
DATA_DIR.mkdir(exist_ok=True)

df[["title", "ingredients", "steps", "text"]].to_csv(
    DATA_DIR / "processed_recipes.csv", index=False, encoding="utf-8"
)

# Reload it to confirm it saved correctly
check = pd.read_csv(DATA_DIR / "processed_recipes.csv")
print("Saved:", (DATA_DIR / "processed_recipes.csv").resolve())
print("Shape:", check.shape)
print(check.columns.tolist())
print(check["text"].iloc[0][:200])

Saved: C:\Users\uwase\Desktop\PROJECTS\MiniChefGPT\data\processed_recipes.csv
Shape: (8000, 4)
['title', 'ingredients', 'steps', 'text']
Title: Delicious Baked Beans
Ingredients:
- bacon
- onions
- garlic
- ginger
- great northern beans
- brown sugar
- molasses
- barbecue sauce
- ketchup
- Worcestershire sauce
- prepared mustard
- cide
